# Genotype–Phenotype Heterogeneity Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIR^2 clinical dataset (3 female patients, NC-21OHD) defined by a Croissant schema, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is specified via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json
```
This dataset contains multiple tabular record sets (clinical features, hormone levels, genotypes), all entities referenced by their unique `@id`.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. This establishes a Python object interface to the Croissant schema and enables exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json"

# Load Croissant schema (metadata)
dataset = mlc.Dataset(croissant_url)

# Access metadata (as an object)
print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Identifier: {dataset.metadata.identifier}")

## 2. Data Overview
Review available record sets, their `@id`s, related fields and columns. All references use the unique `@id` assigned by Croissant.

In [ ]:
# List all record sets, fields, and columns by their Croissant @id

record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        # List fields for this record set
        fields = rs.get('field', [])
        for f in fields:
            print(f"   Field @id: {f['@id']} - name: {f.get('name','N/A')}")
            columns = f.get('column', [])
            for c in columns:
                print(f"      Column @id: {c['@id']} - name: {c.get('name','N/A')}")

## 3. Data Extraction
Load each record set (table) into a DataFrame for analysis. All access is performed via unique `@id`. Substitute or extend to access specific fields or columns.

If no record sets are listed above (in section 2), you may need to directly use known `@id` references or consult the data specification.

In [ ]:
# Example record set @id values (replace with actual @id from section above)
# According to description, 3 tables: clinical features (Table 1), hormone/imaging (Table 2), genetic variants (Table 3)
# These @ids are hypothetical – replace with values found above (e.g., 'cr:RecordSet_Table1', etc.)
record_set_ids = [
    'cr:RecordSet_Table1',  # Table 1
    'cr:RecordSet_Table2',  # Table 2
    'cr:RecordSet_Table3'   # Table 3
]

dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for {rs_id}: columns={df.columns.tolist()}")
    except Exception as e:
        print(f"Error loading {rs_id}: {e}")

# Display the first rows of Table 1 as an example
example_rs = 'cr:RecordSet_Table1'
if example_rs in dataframes:
    display(dataframes[example_rs].head())

## 4. Exploratory Data Analysis (EDA)
Process data from a record set (e.g., Table 2 – hormone levels) using `@id` references.
- Filter numeric field (e.g., serum 17-OHP @id: `cr:17OHP`)
- Normalize field
- Group by categorical variable (e.g., patient identifier @id: `cr:PatientID`)

All entity references are via their Croissant `@id`.

In [ ]:
# Choose Table 2 (hormone/imaging) for EDA
record_set_id = 'cr:RecordSet_Table2'  # Replace with correct @id from overview
numeric_field_id = 'cr:17OHP'         # e.g. serum 17-hydroxyprogesterone
group_field_id = 'cr:PatientID'       # e.g. group by patient

# Only proceed if record set exists
if record_set_id in dataframes:
    df = dataframes[record_set_id]
    # Filter value > threshold
    threshold = 10
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        
        # Normalize numeric field
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Group by patient
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print(f"Numeric field {numeric_field_id} not found in columns: {df.columns}")
else:
    print(f"Record set {record_set_id} not loaded.")

## 5. Visualization
Visualize distributions or relationships using pandas/matplotlib. All fields referenced by `@id`.
Example: plot histogram of hormone (17OHP) from Table 2.

In [ ]:
import matplotlib.pyplot as plt

# Plot distribution for Table 2, 17OHP if available
record_set_id = 'cr:RecordSet_Table2'
numeric_field_id = 'cr:17OHP'
if record_set_id in dataframes and numeric_field_id in dataframes[record_set_id].columns:
    df = dataframes[record_set_id]
    plt.figure(figsize=(6,4))
    plt.hist(df[numeric_field_id].dropna(), bins=5)
    plt.xlabel('Serum 17OHP (ng/dL)')
    plt.ylabel('Count')
    plt.title('Distribution of 17OHP levels (Field @id: cr:17OHP)')
    plt.show()
else:
    print(f"No numeric data for {numeric_field_id} in record set {record_set_id}.")

## 6. Conclusion
Using mlcroissant, we've loaded FAIR^2 clinical datasets by Croissant schema, referencing all entities by their `@id`. Analysis showed serum 17OHP variation and allowed tabulation/normalization.

_For further use, substitute real `@id`s from record sets, fields, and columns as listed in the schema (see section 2)._